In [ ]:
from sagemaker.tensorflow    import TensorFlow
from sagemaker.inputs        import TrainingInput
from sagemaker.tuner         import HyperparameterTuner, ContinuousParameter, IntegerParameter, CategoricalParameter

estimator = TensorFlow(
    entry_point = "tf+sagemaker_model.py",
    framework_version = "2.19",
    py_version = "py312",
    instance_type = "ml.m6i.xlarge",
    instance_count = 1,
    output_path = "s3://{}/{}/output/".format(bucket, prefix),
    model_dir = False,  # disable --model_dir injection (use SM_MODEL_DIR env var)
    role = role
)

tuner = HyperparameterTuner(
    estimator             = estimator,
    objective_metric_name = "val:loss",
    objective_type        = "Minimize",             # "Minimize" | "Maximize"
    hyperparameter_ranges = {
        "lr":           ContinuousParameter(0.00001,  0.01,  scaling_type="Logarithmic"),
        "batch_size":   CategoricalParameter([32, 64]),
        "epochs":       IntegerParameter(10, 20),
    },
    metric_definitions    = [
        {"Name": "train:loss", "Regex": r"loss: ([0-9\.]+)"},
        {"Name": "train:acc",  "Regex": r"accuracy: ([0-9\.]+)"},
        {"Name": "val:loss",   "Regex": r"val_loss: ([0-9\.]+)"},
        {"Name": "val:acc",    "Regex": r"val_accuracy: ([0-9\.]+)"},
    ],
    strategy              = "Bayesian",              # "Bayesian" | "Random" | "Hyperband" | "Grid"
    max_jobs              = 10,
    max_parallel_jobs     = 5,                       # Bayesian은 적게 → 이전 결과를 다음 탐색에 활용
    early_stopping_type   = "Auto",                  # Hyperband 사용 시 반드시 "Off" (내부 stopping 사용)
)
tuner.fit({"train": TrainingInput("s3://{}/{}/dataset.zip".format(bucket, prefix))}, wait=False)

In [ ]:
from sagemaker.serializers   import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer

tuner.wait()
best_estimator = tuner.best_estimator()
predictor      = best_estimator.deploy(
    initial_instance_count = 1,
    instance_type          = "ml.g5.xlarge",
    serializer             = IdentitySerializer(content_type="image/jpeg"),
    deserializer           = JSONDeserializer(),
)